# Task 2 ? Experimental Model 1: BiGRU-Attention Sentiment Classifier

**Member:** Anshika Goel  
**Dataset:** Yelp Polarity  
**Model:** learned embedding ? parallel temporal convolutions ? global max pooling ? dropout ? linear classifier

## Scientific role

This notebook trains the first experimental architecture from scratch
using only:

- the deterministic 504,000-example training subset;
- the deterministic 56,000-example validation subset;
- the vocabulary learned from the training subset only.

The official Yelp Polarity test split is intentionally **not loaded,
inspected, evaluated, or used for model selection in this notebook**.

The test set remains reserved for the common final evaluation after all
three candidate architectures have been trained.

## BiGRU-Attention hypothesis

The baseline masked-mean architecture discards local word order.

BiGRU-Attention instead learns local phrase detectors using three convolutional
kernel widths: **3, 4, and 5**. Each branch therefore searches for
discriminative patterns at a different phrase length.

Global max pooling retains the strongest learned response from each filter
anywhere in the review. This tests whether local phrase structure provides
useful sentiment information beyond unordered mean pooling.

## Model-selection rule

The best checkpoint is selected using **validation macro-F1 at the fixed
decision threshold 0.5**.

If validation macro-F1 ties, lower validation loss is preferred.

Early stopping uses validation performance only.

No pretrained language model or pretrained embedding is used.


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import psutil
import torch
import yaml
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset


def find_repo_root():
    current = Path.cwd().resolve()

    candidates = [
        current,
        *current.parents,
    ]

    for candidate in candidates:
        expected = (
            candidate
            / "task2_sentiment"
            / "Anshika_Goel"
            / "configs"
            / "sentiment.yaml"
        )

        if expected.is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate repository root "
        "containing Task 2 configuration."
    )


def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


repo_root = find_repo_root()

member_root = (
    repo_root
    / "task2_sentiment"
    / "Anshika_Goel"
)

config_path = (
    member_root
    / "configs"
    / "sentiment.yaml"
)

vocabulary_path = (
    member_root
    / "outputs"
    / "metrics"
    / "vocabulary.json"
)

split_manifest_path = (
    member_root
    / "outputs"
    / "metrics"
    / "split_manifest.json"
)

processed_dir = (
    repo_root
    / "task2_sentiment"
    / "data"
    / "processed"
    / "Anshika_Goel"
)

checkpoint_dir = (
    member_root
    / "checkpoints"
)

metrics_dir = (
    member_root
    / "outputs"
    / "metrics"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

metrics_dir.mkdir(
    parents=True,
    exist_ok=True,
)

logs_dir = (
    member_root
    / "logs"
)

logs_dir.mkdir(
    parents=True,
    exist_ok=True,
)

training_log_path = (
    logs_dir
    / "bigru_attention_training_run_001.log"
)


def write_training_log(message=""):
    line = str(message)

    print(line)

    with training_log_path.open(
        "a",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        handle.write(
            line + "\n"
        )

        handle.flush()


config = yaml.safe_load(
    config_path.read_text(
        encoding="utf-8"
    )
)

data_config = config["data"]
training_config = config["training"]

model_config = (
    config["models"][
        "bigru_attention"
    ]
)


training_seed = int(
    training_config["training_seed"]
)

random.seed(training_seed)
np.random.seed(training_seed)
torch.manual_seed(training_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        training_seed
    )

torch.use_deterministic_algorithms(
    True,
    warn_only=True,
)

if torch.backends.cudnn.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

amp_enabled = bool(
    training_config["mixed_precision"]
    and device.type == "cuda"
)


print(
    "Configuration loaded:",
    config_path.relative_to(repo_root),
)

print(
    "Device:",
    device,
)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

print(
    "Automatic mixed precision:",
    amp_enabled,
)

print(
    "Training seed:",
    training_seed,
)

print(
    "OFFICIAL TEST POLICY: "
    "not loaded in this notebook."
)

print(
    "BIGRU-ATTENTION CONFIGURATION VERIFIED"
)

Configuration loaded: task2_sentiment\Anshika_Goel\configs\sentiment.yaml
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Automatic mixed precision: True
Training seed: 20260924
OFFICIAL TEST POLICY: not loaded in this notebook.
BIGRU-ATTENTION CONFIGURATION VERIFIED


## Training and validation data

The encoded arrays were produced by the verified preprocessing notebook.

Important leakage controls:

1. vocabulary construction used only the 504,000-example training subset;
2. the validation subset is used only for model selection and early stopping;
3. the official test set is not accessed here;
4. the decision threshold remains fixed at 0.5 during training;
5. no test-driven hyperparameter adjustment is permitted.

In [2]:
train_inputs_path = (
    processed_dir
    / "train_input_ids.npy"
)

train_labels_path = (
    processed_dir
    / "train_labels.npy"
)

validation_inputs_path = (
    processed_dir
    / "validation_input_ids.npy"
)

validation_labels_path = (
    processed_dir
    / "validation_labels.npy"
)


required_paths = [
    train_inputs_path,
    train_labels_path,
    validation_inputs_path,
    validation_labels_path,
    vocabulary_path,
    split_manifest_path,
]


missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]


if missing_paths:
    raise FileNotFoundError(
        "Required training artifacts are "
        f"missing: {missing_paths}"
    )


vocabulary_artifact = json.loads(
    vocabulary_path.read_text(
        encoding="utf-8"
    )
)

token_to_index = (
    vocabulary_artifact[
        "token_to_index"
    ]
)

vocabulary_size = len(
    token_to_index
)

pad_index = int(
    data_config["pad_index"]
)

unknown_index = int(
    data_config["unknown_index"]
)

sequence_length = int(
    data_config["sequence_length"]
)


assert token_to_index["<PAD>"] == 0
assert token_to_index["<UNK>"] == 1

assert pad_index == 0
assert unknown_index == 1

assert (
    vocabulary_size
    == int(
        data_config[
            "vocabulary_size"
        ]
    )
)


train_inputs = np.load(
    train_inputs_path,
    mmap_mode="r",
    allow_pickle=False,
)

train_labels = np.load(
    train_labels_path,
    mmap_mode="r",
    allow_pickle=False,
)

validation_inputs = np.load(
    validation_inputs_path,
    mmap_mode="r",
    allow_pickle=False,
)

validation_labels = np.load(
    validation_labels_path,
    mmap_mode="r",
    allow_pickle=False,
)


assert train_inputs.shape == (
    504_000,
    sequence_length,
)

assert train_labels.shape == (
    504_000,
)

assert validation_inputs.shape == (
    56_000,
    sequence_length,
)

assert validation_labels.shape == (
    56_000,
)


assert int(
    np.count_nonzero(
        train_labels == 0
    )
) == 252_000

assert int(
    np.count_nonzero(
        train_labels == 1
    )
) == 252_000

assert int(
    np.count_nonzero(
        validation_labels == 0
    )
) == 28_000

assert int(
    np.count_nonzero(
        validation_labels == 1
    )
) == 28_000


print(
    "Training examples:",
    train_inputs.shape[0],
)

print(
    "Validation examples:",
    validation_inputs.shape[0],
)

print(
    "Sequence length:",
    sequence_length,
)

print(
    "Vocabulary size:",
    vocabulary_size,
)

print(
    "Training class counts:",
    {
        "negative": 252_000,
        "positive": 252_000,
    },
)

print(
    "Validation class counts:",
    {
        "negative": 28_000,
        "positive": 28_000,
    },
)

print(
    "TRAINING / VALIDATION DATA VERIFIED"
)

Training examples: 504000
Validation examples: 56000
Sequence length: 256
Vocabulary size: 50000
Training class counts: {'negative': 252000, 'positive': 252000}
Validation class counts: {'negative': 28000, 'positive': 28000}
TRAINING / VALIDATION DATA VERIFIED


In [3]:
class IndexDataset(Dataset):
    def __init__(self, size):
        self.size = int(size)

    def __len__(self):
        return self.size

    def __getitem__(self, index):
        return int(index)


def make_collate_function(
    input_array,
    label_array,
):
    def collate(batch_indices):
        indices = np.asarray(
            batch_indices,
            dtype=np.int64,
        )

        batch_inputs = np.array(
            input_array[indices],
            dtype=np.int64,
            copy=True,
        )

        batch_labels = np.array(
            label_array[indices],
            dtype=np.float32,
            copy=True,
        )

        return (
            torch.from_numpy(
                batch_inputs
            ),
            torch.from_numpy(
                batch_labels
            ),
        )

    return collate


batch_size = int(
    model_config["batch_size"]
)

num_workers = int(
    training_config["num_workers"]
)

pin_memory = (
    device.type == "cuda"
)


train_generator = torch.Generator()
train_generator.manual_seed(
    training_seed
)


train_loader = DataLoader(
    IndexDataset(
        len(train_inputs)
    ),
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    generator=train_generator,
    collate_fn=make_collate_function(
        train_inputs,
        train_labels,
    ),
)


validation_loader = DataLoader(
    IndexDataset(
        len(validation_inputs)
    ),
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    collate_fn=make_collate_function(
        validation_inputs,
        validation_labels,
    ),
)


print(
    "Training batches per epoch:",
    len(train_loader),
)

print(
    "Validation batches:",
    len(validation_loader),
)

print(
    "Batch size:",
    batch_size,
)

print(
    "DataLoader workers:",
    num_workers,
)

print(
    "DATALOADERS CREATED"
)

Training batches per epoch: 3938
Validation batches: 438
Batch size: 128
DataLoader workers: 0
DATALOADERS CREATED


## BiGRU-Attention architecture

Each token ID is mapped to a randomly initialized and trainable
128-dimensional embedding.

The embedding sequence is transposed to the channel-first format expected
by `Conv1d`.

Three independent convolutional branches use:

- kernel size 3 with 128 filters;
- kernel size 4 with 128 filters;
- kernel size 5 with 128 filters.

For every branch:

1. temporal convolution extracts local phrase features;
2. ReLU introduces non-linearity;
3. convolution positions containing no real token are excluded;
4. global max pooling retains the strongest activation from each filter.

The three pooled outputs are concatenated into a 384-dimensional
review-level representation.

Dropout is then applied before the final linear binary-classification layer.

PAD-only sequences are handled explicitly: when no convolution window
contains a real token, the pooled representation for that branch is set to
zero rather than to an extreme floating-point masking value.


In [4]:
class BiGRUAttentionSentimentClassifier(
    nn.Module
):
    def __init__(
        self,
        vocabulary_size,
        embedding_dimension,
        padding_index,
        hidden_dimension,
        recurrent_layers,
        bidirectional,
        attention_dimension,
        dropout_probability,
    ):
        super().__init__()

        self.padding_index = int(
            padding_index
        )

        self.hidden_dimension = int(
            hidden_dimension
        )

        self.recurrent_layers = int(
            recurrent_layers
        )

        self.bidirectional = bool(
            bidirectional
        )

        self.attention_dimension = int(
            attention_dimension
        )

        self.num_directions = (
            2
            if self.bidirectional
            else 1
        )

        self.recurrent_output_dimension = (
            self.hidden_dimension
            * self.num_directions
        )

        self.embedding = nn.Embedding(
            num_embeddings=(
                vocabulary_size
            ),
            embedding_dim=(
                embedding_dimension
            ),
            padding_idx=(
                self.padding_index
            ),
        )

        self.gru = nn.GRU(
            input_size=(
                embedding_dimension
            ),
            hidden_size=(
                self.hidden_dimension
            ),
            num_layers=(
                self.recurrent_layers
            ),
            batch_first=True,
            bidirectional=(
                self.bidirectional
            ),
            dropout=(
                dropout_probability
                if self.recurrent_layers > 1
                else 0.0
            ),
        )

        self.attention_projection = nn.Linear(
            self.recurrent_output_dimension,
            self.attention_dimension,
            bias=True,
        )

        self.attention_score = nn.Linear(
            self.attention_dimension,
            1,
            bias=False,
        )

        self.dropout = nn.Dropout(
            p=dropout_probability
        )

        self.classifier = nn.Linear(
            self.recurrent_output_dimension,
            1,
        )

        self.reset_parameters()


    def reset_parameters(self):
        nn.init.normal_(
            self.embedding.weight,
            mean=0.0,
            std=0.02,
        )

        with torch.no_grad():
            self.embedding.weight[
                self.padding_index
            ].zero_()

        for name, parameter in (
            self.gru.named_parameters()
        ):
            if (
                "weight_ih"
                in name
            ):
                for gate_weight in (
                    parameter.chunk(
                        3,
                        dim=0,
                    )
                ):
                    nn.init.xavier_uniform_(
                        gate_weight
                    )

            elif (
                "weight_hh"
                in name
            ):
                for gate_weight in (
                    parameter.chunk(
                        3,
                        dim=0,
                    )
                ):
                    nn.init.orthogonal_(
                        gate_weight
                    )

            elif (
                "bias"
                in name
            ):
                nn.init.zeros_(
                    parameter
                )

        nn.init.xavier_uniform_(
            self.attention_projection.weight
        )

        nn.init.zeros_(
            self.attention_projection.bias
        )

        nn.init.xavier_uniform_(
            self.attention_score.weight
        )

        nn.init.xavier_uniform_(
            self.classifier.weight
        )

        nn.init.zeros_(
            self.classifier.bias
        )


    def forward(
        self,
        input_ids,
    ):
        token_mask = (
            input_ids
            != self.padding_index
        )

        sequence_lengths = (
            token_mask.sum(
                dim=1
            )
            .to(
                dtype=torch.int64
            )
        )

        packed_lengths = (
            sequence_lengths
            .clamp_min(
                1
            )
            .cpu()
        )

        embedded = self.embedding(
            input_ids
        )

        packed_input = (
            nn.utils.rnn.pack_padded_sequence(
                embedded,
                lengths=packed_lengths,
                batch_first=True,
                enforce_sorted=False,
            )
        )

        packed_output, _ = self.gru(
            packed_input
        )

        recurrent_output, _ = (
            nn.utils.rnn.pad_packed_sequence(
                packed_output,
                batch_first=True,
                total_length=(
                    input_ids.shape[
                        1
                    ]
                ),
            )
        )

        attention_hidden = torch.tanh(
            self.attention_projection(
                recurrent_output
            )
        )

        attention_scores = (
            self.attention_score(
                attention_hidden
            )
            .squeeze(-1)
        )

        minimum_value = torch.finfo(
            attention_scores.dtype
        ).min

        masked_attention_scores = (
            attention_scores.masked_fill(
                ~token_mask,
                minimum_value,
            )
        )

        attention_weights = torch.softmax(
            masked_attention_scores,
            dim=1,
        )

        attention_weights = (
            attention_weights
            * token_mask.to(
                dtype=(
                    attention_weights.dtype
                )
            )
        )

        attention_normalizer = (
            attention_weights.sum(
                dim=1,
                keepdim=True,
            )
        )

        safe_normalizer = torch.where(
            attention_normalizer > 0,
            attention_normalizer,
            torch.ones_like(
                attention_normalizer
            ),
        )

        attention_weights = (
            attention_weights
            / safe_normalizer
        )

        context = torch.bmm(
            attention_weights.unsqueeze(
                1
            ),
            recurrent_output,
        ).squeeze(
            1
        )

        context = self.dropout(
            context
        )

        logits = (
            self.classifier(
                context
            )
            .squeeze(-1)
        )

        return logits


embedding_dimension = int(
    model_config[
        "embedding_dimension"
    ]
)

hidden_dimension = int(
    model_config[
        "hidden_dimension"
    ]
)

recurrent_layers = int(
    model_config[
        "recurrent_layers"
    ]
)

bidirectional = bool(
    model_config[
        "bidirectional"
    ]
)

attention_dimension = int(
    model_config[
        "attention_dimension"
    ]
)

dropout_probability = float(
    model_config[
        "dropout"
    ]
)


assert embedding_dimension == 128
assert hidden_dimension == 128
assert recurrent_layers == 1
assert bidirectional is True
assert attention_dimension == 128
assert dropout_probability == 0.3


model = (
    BiGRUAttentionSentimentClassifier(
        vocabulary_size=(
            vocabulary_size
        ),
        embedding_dimension=(
            embedding_dimension
        ),
        padding_index=(
            pad_index
        ),
        hidden_dimension=(
            hidden_dimension
        ),
        recurrent_layers=(
            recurrent_layers
        ),
        bidirectional=(
            bidirectional
        ),
        attention_dimension=(
            attention_dimension
        ),
        dropout_probability=(
            dropout_probability
        ),
    )
    .to(device)
)


total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


num_directions = (
    2
    if bidirectional
    else 1
)

recurrent_output_dimension = (
    hidden_dimension
    * num_directions
)


expected_parameter_count = (
    vocabulary_size
    * embedding_dimension
)


for layer_index in range(
    recurrent_layers
):
    layer_input_dimension = (
        embedding_dimension
        if layer_index == 0
        else recurrent_output_dimension
    )

    expected_parameter_count += (
        num_directions
        * (
            (
                3
                * hidden_dimension
                * layer_input_dimension
            )
            + (
                3
                * hidden_dimension
                * hidden_dimension
            )
            + (
                6
                * hidden_dimension
            )
        )
    )


expected_parameter_count += (
    recurrent_output_dimension
    * attention_dimension
)

expected_parameter_count += (
    attention_dimension
)

expected_parameter_count += (
    attention_dimension
)

expected_parameter_count += (
    recurrent_output_dimension
    + 1
)


assert expected_parameter_count == 6_631_425

assert (
    total_parameters
    == expected_parameter_count
)

assert (
    trainable_parameters
    == expected_parameter_count
)


print(model)

print(
    "Total parameters:",
    f"{total_parameters:,}",
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}",
)

print(
    "Embedding dimension:",
    embedding_dimension,
)

print(
    "Hidden dimension:",
    hidden_dimension,
)

print(
    "Recurrent layers:",
    recurrent_layers,
)

print(
    "Bidirectional:",
    bidirectional,
)

print(
    "Recurrent output dimension:",
    recurrent_output_dimension,
)

print(
    "Attention dimension:",
    attention_dimension,
)

print(
    "Dropout:",
    dropout_probability,
)

print(
    "BIGRU-ATTENTION MODEL CREATED"
)


BiGRUAttentionSentimentClassifier(
  (embedding): Embedding(50000, 128, padding_idx=0)
  (gru): GRU(128, 128, batch_first=True, bidirectional=True)
  (attention_projection): Linear(in_features=256, out_features=128, bias=True)
  (attention_score): Linear(in_features=128, out_features=1, bias=False)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)
Total parameters: 6,631,425
Trainable parameters: 6,631,425
Embedding dimension: 128
Hidden dimension: 128
Recurrent layers: 1
Bidirectional: True
Recurrent output dimension: 256
Attention dimension: 128
Dropout: 0.3
BIGRU-ATTENTION MODEL CREATED


In [5]:
def expected_calibration_error(
    y_true,
    probabilities,
    bins=15,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        bins + 1,
    )

    predicted_labels = (
        probabilities >= 0.5
    ).astype(np.int64)

    confidences = np.maximum(
        probabilities,
        1.0 - probabilities,
    )

    correctness = (
        predicted_labels == y_true
    ).astype(np.float64)

    ece = 0.0

    for bin_index in range(bins):
        lower = bin_edges[
            bin_index
        ]

        upper = bin_edges[
            bin_index + 1
        ]

        if bin_index == bins - 1:
            mask = (
                (confidences >= lower)
                & (confidences <= upper)
            )
        else:
            mask = (
                (confidences >= lower)
                & (confidences < upper)
            )

        if not np.any(mask):
            continue

        bin_fraction = float(
            np.mean(mask)
        )

        bin_accuracy = float(
            np.mean(
                correctness[mask]
            )
        )

        bin_confidence = float(
            np.mean(
                confidences[mask]
            )
        )

        ece += (
            bin_fraction
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(ece)


def classification_metrics(
    labels,
    probabilities,
    threshold=0.5,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "macro_precision": float(
            macro_precision
        ),
        "macro_recall": float(
            macro_recall
        ),
        "macro_f1": float(
            macro_f1
        ),
        "mcc": float(
            matthews_corrcoef(
                labels,
                predictions,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                labels,
                probabilities,
            )
        ),
        "ece_15_bins": float(
            expected_calibration_error(
                labels,
                probabilities,
                bins=15,
            )
        ),
        "threshold": float(
            threshold
        ),
    }


criterion = (
    nn.BCEWithLogitsLoss()
)


@torch.inference_mode()
def evaluate_validation(
    model,
    loader,
):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_labels = []
    all_probabilities = []

    start_time = (
        time.perf_counter()
    )

    for (
        batch_inputs,
        batch_labels,
    ) in loader:
        batch_inputs = (
            batch_inputs.to(
                device,
                non_blocking=pin_memory,
            )
        )

        batch_labels = (
            batch_labels.to(
                device,
                non_blocking=pin_memory,
            )
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logits = model(
                batch_inputs
            )

            loss = criterion(
                logits,
                batch_labels,
            )

        probabilities = (
            torch.sigmoid(
                logits
            )
        )

        batch_count = int(
            batch_labels.shape[0]
        )

        total_loss += (
            float(loss.item())
            * batch_count
        )

        total_examples += (
            batch_count
        )

        all_labels.append(
            batch_labels
            .detach()
            .cpu()
            .numpy()
        )

        all_probabilities.append(
            probabilities
            .detach()
            .cpu()
            .numpy()
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = (
        time.perf_counter()
        - start_time
    )

    labels = np.concatenate(
        all_labels
    )

    probabilities = (
        np.concatenate(
            all_probabilities
        )
    )

    metrics = (
        classification_metrics(
            labels,
            probabilities,
            threshold=float(
                training_config[
                    "decision_threshold"
                ]
            ),
        )
    )

    metrics[
        "loss"
    ] = float(
        total_loss
        / total_examples
    )

    metrics[
        "examples_per_second"
    ] = float(
        total_examples
        / elapsed
    )

    metrics[
        "evaluation_seconds"
    ] = float(
        elapsed
    )

    return metrics


print(
    "VALIDATION METRIC FUNCTIONS DEFINED"
)

VALIDATION METRIC FUNCTIONS DEFINED


In [6]:
maximum_epochs = int(
    model_config[
        "maximum_epochs"
    ]
)

learning_rate = float(
    model_config[
        "learning_rate"
    ]
)

weight_decay = float(
    model_config[
        "weight_decay"
    ]
)

warmup_ratio = float(
    training_config[
        "warmup_ratio"
    ]
)

minimum_learning_rate = float(
    training_config[
        "minimum_learning_rate"
    ]
)

gradient_clip_norm = float(
    training_config[
        "gradient_clip_norm"
    ]
)

early_stopping_patience = int(
    training_config[
        "early_stopping_patience"
    ]
)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)


total_training_steps = (
    maximum_epochs
    * len(train_loader)
)

warmup_steps = int(
    round(
        total_training_steps
        * warmup_ratio
    )
)

minimum_lr_factor = (
    minimum_learning_rate
    / learning_rate
)


def learning_rate_multiplier(
    step,
):
    if (
        warmup_steps > 0
        and step < warmup_steps
    ):
        return max(
            minimum_lr_factor,
            float(step + 1)
            / float(
                warmup_steps
            ),
        )

    decay_steps = max(
        1,
        total_training_steps
        - warmup_steps,
    )

    progress = min(
        1.0,
        max(
            0.0,
            float(
                step
                - warmup_steps
            )
            / float(
                decay_steps
            ),
        ),
    )

    return max(
        minimum_lr_factor,
        1.0
        - progress
        * (
            1.0
            - minimum_lr_factor
        ),
    )


scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=(
            learning_rate_multiplier
        ),
    )
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled,
)


best_checkpoint_path = (
    checkpoint_dir
    / "bigru_attention_best.pt"
)

temporary_checkpoint_path = (
    checkpoint_dir
    / "bigru_attention_best.pt.tmp"
)

history_path = (
    metrics_dir
    / "bigru_attention_history.json"
)

metrics_path = (
    metrics_dir
    / "bigru_attention_metrics.json"
)


for output_path in [
    best_checkpoint_path,
    temporary_checkpoint_path,
    history_path,
    metrics_path,
    training_log_path,
]:
    if output_path.exists():
        raise FileExistsError(
            "Refusing to overwrite "
            f"existing BiGRU-Attention artifact: "
            f"{output_path.relative_to(repo_root)}"
        )


print(
    "Optimizer:",
    optimizer.__class__.__name__,
)

print(
    "Initial learning rate:",
    learning_rate,
)

print(
    "Minimum learning rate:",
    minimum_learning_rate,
)

print(
    "Weight decay:",
    weight_decay,
)

print(
    "Maximum epochs:",
    maximum_epochs,
)

print(
    "Warmup steps:",
    warmup_steps,
)

print(
    "Total planned steps:",
    total_training_steps,
)

print(
    "Early stopping patience:",
    early_stopping_patience,
)

print(
    "Model selection metric:",
    "validation macro-F1",
)

print(
    "TRAINING CONFIGURATION READY"
)

Optimizer: AdamW
Initial learning rate: 0.0005
Minimum learning rate: 1e-05
Weight decay: 0.0001
Maximum epochs: 5
Warmup steps: 984
Total planned steps: 19690
Early stopping patience: 2
Model selection metric: validation macro-F1
TRAINING CONFIGURATION READY


## Training

The training loop below:

- uses binary cross-entropy with logits;
- uses AdamW;
- applies gradient clipping;
- follows the configured warmup/decay schedule;
- uses mixed precision only when CUDA is available;
- evaluates only the validation split after each epoch;
- saves the checkpoint with the highest validation macro-F1;
- stops after the configured number of consecutive non-improving epochs.

The raw notebook execution log will be captured externally when this
notebook is executed so the training evidence remains unedited.

### Raw training evidence

In addition to notebook outputs, the training loop writes each
training event directly to
`logs/bigru_attention_training_run_001.log` as it occurs.

The log is append-and-flush at every recorded event, and the notebook
refuses to overwrite an existing run-001 log. This preserves the raw
epoch-level training evidence independently of notebook rendering.


In [7]:
history = []

best_macro_f1 = -1.0
best_validation_loss = float(
    "inf"
)

best_epoch = None
best_validation_metrics = None

epochs_without_improvement = 0

training_start_utc = (
    datetime.now(
        timezone.utc
    ).isoformat()
)

overall_start_time = (
    time.perf_counter()
)


write_training_log(
    "BIGRU-ATTENTION TRAINING RAW LOG"
)

write_training_log(
    f"training_start_utc={training_start_utc}"
)

write_training_log(
    "model=bigru_attention"
)

write_training_log(
    f"architecture={model_config['architecture']}"
)

write_training_log(
    f"training_seed={training_seed}"
)

write_training_log(
    f"device={device}"
)

write_training_log(
    "gpu="
    + (
        torch.cuda.get_device_name(0)
        if device.type == "cuda"
        else "None"
    )
)

write_training_log(
    f"mixed_precision={amp_enabled}"
)

write_training_log(
    f"training_examples={len(train_inputs)}"
)

write_training_log(
    f"validation_examples={len(validation_inputs)}"
)

write_training_log(
    f"batch_size={batch_size}"
)

write_training_log(
    f"maximum_epochs={maximum_epochs}"
)

write_training_log(
    f"learning_rate={learning_rate}"
)

write_training_log(
    f"minimum_learning_rate={minimum_learning_rate}"
)

write_training_log(
    f"weight_decay={weight_decay}"
)

write_training_log(
    f"warmup_ratio={warmup_ratio}"
)

write_training_log(
    f"gradient_clip_norm={gradient_clip_norm}"
)

write_training_log(
    f"early_stopping_patience={early_stopping_patience}"
)

write_training_log(
    "model_selection_metric=validation_macro_f1"
)

write_training_log(
    f"decision_threshold={training_config['decision_threshold']}"
)

write_training_log(
    "official_test_used=False"
)

write_training_log(
    ""
)


if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


for epoch in range(
    1,
    maximum_epochs + 1,
):
    model.train()

    epoch_loss_sum = 0.0
    epoch_example_count = 0

    if device.type == "cuda":
        torch.cuda.synchronize()

    epoch_start_time = (
        time.perf_counter()
    )


    for (
        batch_inputs,
        batch_labels,
    ) in train_loader:
        batch_inputs = (
            batch_inputs.to(
                device,
                non_blocking=pin_memory,
            )
        )

        batch_labels = (
            batch_labels.to(
                device,
                non_blocking=pin_memory,
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logits = model(
                batch_inputs
            )

            loss = criterion(
                logits,
                batch_labels,
            )


        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=gradient_clip_norm,
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        scheduler.step()


        batch_count = int(
            batch_labels.shape[0]
        )

        epoch_loss_sum += (
            float(loss.item())
            * batch_count
        )

        epoch_example_count += (
            batch_count
        )


    if device.type == "cuda":
        torch.cuda.synchronize()


    epoch_training_seconds = (
        time.perf_counter()
        - epoch_start_time
    )

    epoch_train_loss = (
        epoch_loss_sum
        / epoch_example_count
    )

    epoch_examples_per_second = (
        epoch_example_count
        / epoch_training_seconds
    )


    validation_metrics = (
        evaluate_validation(
            model,
            validation_loader,
        )
    )


    current_macro_f1 = float(
        validation_metrics[
            "macro_f1"
        ]
    )

    current_validation_loss = (
        float(
            validation_metrics[
                "loss"
            ]
        )
    )


    improved = (
        current_macro_f1
        > best_macro_f1
        + 1e-12
    )

    tied_macro_f1 = (
        abs(
            current_macro_f1
            - best_macro_f1
        )
        <= 1e-12
    )

    if (
        tied_macro_f1
        and current_validation_loss
        < best_validation_loss
    ):
        improved = True


    epoch_record = {
        "epoch": int(
            epoch
        ),
        "train_loss": float(
            epoch_train_loss
        ),
        "train_examples_per_second": float(
            epoch_examples_per_second
        ),
        "train_seconds": float(
            epoch_training_seconds
        ),
        "learning_rate_end": float(
            optimizer.param_groups[0][
                "lr"
            ]
        ),
        "validation": (
            validation_metrics
        ),
        "checkpoint_improved": bool(
            improved
        ),
    }


    history.append(
        epoch_record
    )


    if improved:
        best_macro_f1 = (
            current_macro_f1
        )

        best_validation_loss = (
            current_validation_loss
        )

        best_epoch = int(
            epoch
        )

        best_validation_metrics = dict(
            validation_metrics
        )

        checkpoint_payload = {
            "model_name": (
                "bigru_attention"
            ),
            "architecture": (
                model_config[
                    "architecture"
                ]
            ),
            "model_state_dict": (
                model.state_dict()
            ),
            "best_epoch": (
                best_epoch
            ),
            "best_validation_metrics": (
                best_validation_metrics
            ),
            "vocabulary_size": int(
                vocabulary_size
            ),
            "embedding_dimension": int(
                embedding_dimension
            ),
            "hidden_dimension": int(
                hidden_dimension
            ),
            "recurrent_layers": int(
                recurrent_layers
            ),
            "bidirectional": bool(
                bidirectional
            ),
            "attention_dimension": int(
                attention_dimension
            ),
            "pad_index": int(
                pad_index
            ),
            "dropout": float(
                dropout_probability
            ),
            "training_seed": int(
                training_seed
            ),
            "decision_threshold": float(
                training_config[
                    "decision_threshold"
                ]
            ),
            "config_sha256": (
                sha256_file(
                    config_path
                )
            ),
            "vocabulary_sha256": (
                sha256_file(
                    vocabulary_path
                )
            ),
            "split_manifest_sha256": (
                sha256_file(
                    split_manifest_path
                )
            ),
        }


        torch.save(
            checkpoint_payload,
            temporary_checkpoint_path,
        )

        os.replace(
            temporary_checkpoint_path,
            best_checkpoint_path,
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1


    history_path.write_text(
        json.dumps(
            {
                "model": (
                    "bigru_attention"
                ),
                "selection_metric": (
                    "validation_macro_f1"
                ),
                "decision_threshold": float(
                    training_config[
                        "decision_threshold"
                    ]
                ),
                "history": history,
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )


    epoch_log_line = (
        f"Epoch {epoch}/{maximum_epochs} | "
        f"train_loss={epoch_train_loss:.6f} | "
        f"val_loss={current_validation_loss:.6f} | "
        f"val_accuracy={validation_metrics['accuracy']:.6f} | "
        f"val_macro_f1={current_macro_f1:.6f} | "
        f"train_examples_per_second="
        f"{epoch_examples_per_second:.2f} | "
        f"checkpoint_improved={improved}"
    )

    write_training_log(
        epoch_log_line
    )


    if (
        epochs_without_improvement
        >= early_stopping_patience
    ):
        write_training_log(
            "Early stopping triggered after "
            f"{epochs_without_improvement} "
            "consecutive non-improving epochs."
        )

        break


if device.type == "cuda":
    torch.cuda.synchronize()


total_training_seconds = (
    time.perf_counter()
    - overall_start_time
)

training_end_utc = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


assert best_checkpoint_path.is_file()
assert best_epoch is not None


print()
print(
    "TRAINING COMPLETED"
)

print(
    "Epochs executed:",
    len(history),
)

print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation macro-F1:",
    best_macro_f1,
)

print(
    "Total training wall seconds:",
    total_training_seconds,
)

write_training_log(
    ""
)

write_training_log(
    "TRAINING COMPLETED"
)

write_training_log(
    f"epochs_executed={len(history)}"
)

write_training_log(
    f"best_epoch={best_epoch}"
)

write_training_log(
    f"best_validation_macro_f1={best_macro_f1}"
)

write_training_log(
    f"total_training_wall_seconds={total_training_seconds}"
)

write_training_log(
    f"training_end_utc={training_end_utc}"
)


BIGRU-ATTENTION TRAINING RAW LOG
training_start_utc=2026-09-25T23:56:02.826727+00:00
model=bigru_attention
architecture=learned_embedding_bidirectional_gru_attention
training_seed=20260924
device=cuda
gpu=NVIDIA GeForce RTX 3060 Laptop GPU
mixed_precision=True
training_examples=504000
validation_examples=56000
batch_size=128
maximum_epochs=5
learning_rate=0.0005
minimum_learning_rate=1e-05
weight_decay=0.0001
warmup_ratio=0.05
gradient_clip_norm=1.0
early_stopping_patience=2
model_selection_metric=validation_macro_f1
decision_threshold=0.5
official_test_used=False



Epoch 1/5 | train_loss=0.209165 | val_loss=0.155350 | val_accuracy=0.939143 | val_macro_f1=0.939133 | train_examples_per_second=1764.39 | checkpoint_improved=True


Epoch 2/5 | train_loss=0.127089 | val_loss=0.147463 | val_accuracy=0.943518 | val_macro_f1=0.943514 | train_examples_per_second=1519.09 | checkpoint_improved=True


Epoch 3/5 | train_loss=0.100535 | val_loss=0.152814 | val_accuracy=0.942214 | val_macro_f1=0.942212 | train_examples_per_second=975.83 | checkpoint_improved=False


Epoch 4/5 | train_loss=0.078079 | val_loss=0.163003 | val_accuracy=0.939839 | val_macro_f1=0.939835 | train_examples_per_second=1345.18 | checkpoint_improved=False
Early stopping triggered after 2 consecutive non-improving epochs.

TRAINING COMPLETED
Epochs executed: 4
Best epoch: 2
Best validation macro-F1: 0.9435136202140673
Total training wall seconds: 1545.914335199981

TRAINING COMPLETED
epochs_executed=4
best_epoch=2
best_validation_macro_f1=0.9435136202140673
total_training_wall_seconds=1545.914335199981
training_end_utc=2026-09-26T00:21:48.741580+00:00


In [8]:
best_checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


final_validation_metrics = (
    evaluate_validation(
        model,
        validation_loader,
    )
)


if device.type == "cuda":
    peak_gpu_allocated_mb = float(
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )

    peak_gpu_reserved_mb = float(
        torch.cuda.max_memory_reserved()
        / (1024 ** 2)
    )

    gpu_name = (
        torch.cuda.get_device_name(0)
    )

    gpu_total_memory_mb = float(
        torch.cuda.get_device_properties(
            0
        ).total_memory
        / (1024 ** 2)
    )

else:
    peak_gpu_allocated_mb = None
    peak_gpu_reserved_mb = None
    gpu_name = None
    gpu_total_memory_mb = None


process = psutil.Process(
    os.getpid()
)

process_rss_mb = float(
    process.memory_info().rss
    / (1024 ** 2)
)


metrics_artifact = {
    "model": (
        "bigru_attention"
    ),
    "architecture": (
        model_config[
            "architecture"
        ]
    ),
    "evaluation_scope": (
        "training_and_validation_only"
    ),
    "official_test_used": False,
    "official_test_policy": (
        "Reserved for common final evaluation "
        "after all three models are trained."
    ),
    "model_selection": {
        "primary_metric": (
            "validation_macro_f1"
        ),
        "tie_breaker": (
            "lower_validation_loss"
        ),
        "decision_threshold": float(
            training_config[
                "decision_threshold"
            ]
        ),
        "early_stopping_patience": int(
            early_stopping_patience
        ),
    },
    "data": {
        "training_examples": int(
            len(train_inputs)
        ),
        "validation_examples": int(
            len(validation_inputs)
        ),
        "sequence_length": int(
            sequence_length
        ),
        "vocabulary_size": int(
            vocabulary_size
        ),
        "pad_index": int(
            pad_index
        ),
        "unknown_index": int(
            unknown_index
        ),
    },
    "model_hyperparameters": {
        "embedding_dimension": int(
            embedding_dimension
        ),
        "hidden_dimension": int(
            hidden_dimension
        ),
        "recurrent_layers": int(
            recurrent_layers
        ),
        "bidirectional": bool(
            bidirectional
        ),
        "attention_dimension": int(
            attention_dimension
        ),
        "dropout": float(
            dropout_probability
        ),
        "batch_size": int(
            batch_size
        ),
        "maximum_epochs": int(
            maximum_epochs
        ),
        "learning_rate": float(
            learning_rate
        ),
        "minimum_learning_rate": float(
            minimum_learning_rate
        ),
        "weight_decay": float(
            weight_decay
        ),
        "warmup_ratio": float(
            warmup_ratio
        ),
        "gradient_clip_norm": float(
            gradient_clip_norm
        ),
    },
    "parameter_count": {
        "total": int(
            total_parameters
        ),
        "trainable": int(
            trainable_parameters
        ),
    },
    "training": {
        "training_seed": int(
            training_seed
        ),
        "mixed_precision": bool(
            amp_enabled
        ),
        "epochs_executed": int(
            len(history)
        ),
        "best_epoch": int(
            best_epoch
        ),
        "training_start_utc": (
            training_start_utc
        ),
        "training_end_utc": (
            training_end_utc
        ),
        "total_training_wall_seconds": float(
            total_training_seconds
        ),
        "mean_epoch_train_examples_per_second": float(
            np.mean(
                [
                    item[
                        "train_examples_per_second"
                    ]
                    for item in history
                ]
            )
        ),
    },
    "best_validation_metrics": (
        best_validation_metrics
    ),
    "best_checkpoint_validation_recheck": (
        final_validation_metrics
    ),
    "memory": {
        "peak_gpu_allocated_mb": (
            peak_gpu_allocated_mb
        ),
        "peak_gpu_reserved_mb": (
            peak_gpu_reserved_mb
        ),
        "process_rss_mb_after_training": (
            process_rss_mb
        ),
    },
    "hardware": {
        "operating_system": (
            platform.platform()
        ),
        "cpu": (
            platform.processor()
        ),
        "logical_processors": int(
            psutil.cpu_count(
                logical=True
            )
        ),
        "physical_processors": (
            psutil.cpu_count(
                logical=False
            )
        ),
        "system_ram_gb": float(
            psutil.virtual_memory().total
            / (1024 ** 3)
        ),
        "gpu": gpu_name,
        "gpu_total_memory_mb": (
            gpu_total_memory_mb
        ),
    },
    "software": {
        "python": (
            platform.python_version()
        ),
        "torch": (
            torch.__version__
        ),
        "torch_cuda_runtime": (
            torch.version.cuda
        ),
    },
    "integrity": {
        "config_sha256": (
            sha256_file(
                config_path
            )
        ),
        "vocabulary_sha256": (
            sha256_file(
                vocabulary_path
            )
        ),
        "split_manifest_sha256": (
            sha256_file(
                split_manifest_path
            )
        ),
        "checkpoint_sha256": (
            sha256_file(
                best_checkpoint_path
            )
        ),
    },
}


metrics_path.write_text(
    json.dumps(
        metrics_artifact,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)


validation_macro_f1_difference = abs(
    float(
        best_validation_metrics[
            "macro_f1"
        ]
    )
    - float(
        final_validation_metrics[
            "macro_f1"
        ]
    )
)

assert (
    validation_macro_f1_difference
    < 1e-10
)


print(
    "Best checkpoint:",
    best_checkpoint_path.relative_to(
        repo_root
    ),
)

print(
    "History:",
    history_path.relative_to(
        repo_root
    ),
)

print(
    "Training metrics:",
    metrics_path.relative_to(
        repo_root
    ),
)

print(
    "Best checkpoint SHA256:",
    metrics_artifact[
        "integrity"
    ][
        "checkpoint_sha256"
    ],
)

print(
    "Best validation macro-F1:",
    final_validation_metrics[
        "macro_f1"
    ],
)

print(
    "Peak GPU allocated MB:",
    peak_gpu_allocated_mb,
)

print(
    "Peak GPU reserved MB:",
    peak_gpu_reserved_mb,
)

print()
print(
    "BIGRU-ATTENTION TRAINING ARTIFACTS WRITTEN"
)

print(
    "OFFICIAL TEST SET REMAINS UNTOUCHED"
)

write_training_log(
    ""
)

write_training_log(
    "BEST CHECKPOINT VALIDATION RECHECK COMPLETED"
)

write_training_log(
    "best_checkpoint="
    + str(
        best_checkpoint_path.relative_to(
            repo_root
        )
    )
)

write_training_log(
    "best_checkpoint_sha256="
    + metrics_artifact[
        "integrity"
    ][
        "checkpoint_sha256"
    ]
)

write_training_log(
    "best_validation_macro_f1_recheck="
    + str(
        final_validation_metrics[
            "macro_f1"
        ]
    )
)

write_training_log(
    "peak_gpu_allocated_mb="
    + str(
        peak_gpu_allocated_mb
    )
)

write_training_log(
    "peak_gpu_reserved_mb="
    + str(
        peak_gpu_reserved_mb
    )
)

write_training_log(
    "BIGRU-ATTENTION TRAINING ARTIFACTS WRITTEN"
)

write_training_log(
    "OFFICIAL TEST SET REMAINS UNTOUCHED"
)


Best checkpoint: task2_sentiment\Anshika_Goel\checkpoints\bigru_attention_best.pt
History: task2_sentiment\Anshika_Goel\outputs\metrics\bigru_attention_history.json
Training metrics: task2_sentiment\Anshika_Goel\outputs\metrics\bigru_attention_metrics.json
Best checkpoint SHA256: da6841121a69416601fbc9fde5e2e9c32fb4cc5da4cd4a9efb758e35ab66657a
Best validation macro-F1: 0.9435136202140673
Peak GPU allocated MB: 223.447265625
Peak GPU reserved MB: 486.0

BIGRU-ATTENTION TRAINING ARTIFACTS WRITTEN
OFFICIAL TEST SET REMAINS UNTOUCHED

BEST CHECKPOINT VALIDATION RECHECK COMPLETED
best_checkpoint=task2_sentiment\Anshika_Goel\checkpoints\bigru_attention_best.pt
best_checkpoint_sha256=da6841121a69416601fbc9fde5e2e9c32fb4cc5da4cd4a9efb758e35ab66657a
best_validation_macro_f1_recheck=0.9435136202140673
peak_gpu_allocated_mb=223.447265625
peak_gpu_reserved_mb=486.0
BIGRU-ATTENTION TRAINING ARTIFACTS WRITTEN
OFFICIAL TEST SET REMAINS UNTOUCHED


## Completion criteria

This notebook is complete only when:

- all cells execute without errors;
- the best BiGRU-Attention validation checkpoint exists;
- the BiGRU-Attention training-history JSON exists;
- the BiGRU-Attention validation-only metrics JSON exists;
- the saved best checkpoint exactly reproduces the recorded validation metrics;
- runtime, throughput, parameter count, and memory use are recorded;
- the raw BiGRU-Attention run-001 training log is preserved;
- the official test set remains unused.

Final official-test metrics, bootstrap confidence intervals, McNemar tests,
slice analysis, confusion matrices, calibration analysis, and prediction
files belong in the later common evaluation workflow.
